# Homework 1 — Scientific Paper Processing with spaCy

This notebook implements a complete NLP pipeline that transforms scientific PDF documents into structured linguistic data. The pipeline processes all nine supplied papers with the same code:

**PDF → Text → Sections → Paragraphs → Sentences → Tokens → Linguistic Annotations**

Each task builds on the previous one, and the final output is a nested JSON file per paper suitable for Homework 2.

In [1]:
import sys
from pathlib import Path

# Resolve project root whether running from notebooks/ or project root
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source path added: {PROJECT_ROOT / 'src'}")

Project root: /Users/hilalhocaoglu/Desktop/nlp-homework-1-scientific-paper-processing
Source path added: /Users/hilalhocaoglu/Desktop/nlp-homework-1-scientific-paper-processing/src


---
## Task 1 — PDF Text Extraction

We use **PyMuPDF** to extract text page by page. The `sort_blocks=True` parameter sorts text blocks by their position on the page, which helps with reading order on single-column pages but can still mix columns on two-column layouts.

The extraction preserves:
- **Page boundaries** — each page becomes a separate record
- **Line structure** — newlines from the PDF are kept
- **Paragraphs** — blank lines between text blocks are preserved

In [2]:
from scipaper.extraction import extract_pdf, document_statistics, save_extracted_text

PAPER_DIR = PROJECT_ROOT / 'data' / 'papers'
PDF_FILES = sorted(PAPER_DIR.glob('*.pdf'))
SORT_BLOCKS = True
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'extracted_text'

missing = [str(p) for p in PDF_FILES if not p.is_file()]
if missing:
    print('WARNING — PDF paths not found:', missing)
else:
    print(f'{len(PDF_FILES)} PDFs ready from {PAPER_DIR}')
    for p in PDF_FILES:
        print(f'  • {p.name}')

9 PDFs ready from /Users/hilalhocaoglu/Desktop/nlp-homework-1-scientific-paper-processing/data/papers
  • 1406.1078v3.pdf
  • 1409.0473v7.pdf
  • 1412.3555v1.pdf
  • 1601.06733v7.pdf
  • 1607.06450v1.pdf
  • 1610.02357v3.pdf
  • 1703.03906v2.pdf
  • AttentionYouNeed.pdf
  • N16-1024.pdf


In [3]:
# Summarize extraction counts for all PDFs in a compact table
import pandas as pd

results = []
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    text_path = OUTPUT_DIR / f'{pdf_path.stem}.txt'
    save_extracted_text(pages, text_path)
    stats = document_statistics(pages)
    results.append({'paper': pdf_path.stem, **stats, 'text_file': str(text_path)})

extraction_table = pd.DataFrame([
    {
        'Paper': row['paper'],
        'Pages': row['pages'],
        'Characters': row['characters'],
        'Words': row['words'],
    }
    for row in results
])
total_row = pd.DataFrame([{
    'Paper': 'TOTAL',
    'Pages': extraction_table['Pages'].sum(),
    'Characters': extraction_table['Characters'].sum(),
    'Words': extraction_table['Words'].sum(),
}])
extraction_table = pd.concat([extraction_table, total_row], ignore_index=True)
display(extraction_table)


,Paper,Pages,Characters,Words
0,1406.1078v3,15,66324,7804
1,1409.0473v7,15,60133,8311
2,1412.3555v1,9,28097,3967
3,1601.06733v7,11,58932,7024
4,1607.06450v1,14,62658,7698
5,1610.02357v3,8,38972,4474
6,1703.03906v2,9,47065,5424
7,AttentionYouNeed,15,44623,6240
8,N16-1024,11,58929,6784
9,TOTAL,107,465733,57726


In [4]:
# Inspect raw extracted text from three pages of the main paper
from html import escape
from IPython.display import HTML, display

pages = extract_pdf(PAPER_DIR / 'AttentionYouNeed.pdf', sort_blocks=SORT_BLOCKS)
for page_idx in [0, 2, 5]:  # pages 1, 3, 6
    page = pages[page_idx]
    excerpt = escape(page['text'][:1200])
    display(HTML(
        f"<h4>Raw extraction preview — page {page['page']}</h4>"
        f"<pre style='white-space:pre-wrap; overflow-wrap:anywhere; "
        f"max-height:28rem; overflow:auto; padding:0.8rem; "
        f"border:1px solid #bbb'>{excerpt}</pre>"
    ))


### Discussion: PDF Extraction Problems

When extracting text from scientific papers, at least five significant problems occur:

1. **Two-column layouts:** PyMuPDF's block sorting can interleave text from the left and right columns, producing unreadable output. For example, on a two-column page, a sentence from the left column may be followed by a sentence from the right column's header rather than the next paragraph.

2. **Mathematical equations:** Equations like `Attention(Q, K, V) = softmax(QK^T / √d_k)V` lose their spatial layout and become fragmented text. Subscripts, superscripts, and special operators (∑, ∏, √) are extracted as isolated Unicode characters without their mathematical context.

3. **Headers and footers:** Running headers (e.g., the conference name or paper title repeated on each page) and page numbers appear mixed into the body text. For instance, `"Proceedings of NIPS 2017"` may appear between two paragraphs of Section 3.

4. **Hyphenation at line breaks:** Words split across lines (e.g., `"atten-\ntion"`) are not automatically rejoined by the PDF extractor, resulting in tokens like `"atten-"` and `"tion"` being treated as separate words.

5. **Tables:** Table structure is lost during extraction. Cell contents are concatenated into lines of text, making it impossible to reconstruct the original rows and columns. A results table may appear as `"28.4 3.5 days 8 GPUs"` without column headers.

The sample excerpts below are shown as wrapped raw text so line breaks remain
visible without making the notebook excessively wide. They are intentionally
not forced into a table: PDF extraction does not reliably preserve table-cell
structure, and the fragments illustrate that limitation.


---
## Task 2 — Section Detection

The section detector combines multiple heuristics to identify headings across all papers without paper-specific rules:

- **Numbered heading regex** — matches patterns like `1`, `3.2`, `A.1`
- **Typography cues** — checks for bold fonts, larger-than-body font size
- **Small-caps fallback** — detects uppercase headings common in conference templates
- **Common unnumbered labels** — recognizes `Abstract`, `References`, `Acknowledgments`

The detector uses PyMuPDF's `dict` output to access font metadata, and sorts candidates by reading order (left column first on two-column pages).

In [5]:
from scipaper.sections import detect_sections
import json
import pandas as pd

# Detect sections for all papers
all_sections = {}
section_rows = []
for pdf_path in PDF_FILES:
    sections = detect_sections(pdf_path)
    all_sections[pdf_path.stem] = sections
    section_rows.append({"Paper": pdf_path.stem, "Section candidates": len(sections)})

section_rows.append({
    "Paper": "TOTAL",
    "Section candidates": sum(r["Section candidates"] for r in section_rows),
})
display(pd.DataFrame(section_rows))


,Paper,Section candidates
0,1406.1078v3,23
1,1409.0473v7,33
2,1412.3555v1,14
3,1601.06733v7,14
4,1607.06450v1,22
5,1610.02357v3,19
6,1703.03906v2,20
7,AttentionYouNeed,25
8,N16-1024,22
9,TOTAL,192


In [6]:
# Show sections detected in 'Attention Is All You Need' as a table
import pandas as pd

rows = []
for sec in all_sections.get("AttentionYouNeed", []):
    rows.append({
        "ID": sec["section_id"] or "",
        "Title": sec["title"],
        "Page": sec["page"],
    })
print("Sections detected in AttentionYouNeed.pdf:")
display(pd.DataFrame(rows))


Sections detected in AttentionYouNeed.pdf:


,ID,Title,Page
0,,Abstract,1
1,1,Introduction,2
2,2,Background,2
3,3,Model Architecture,2
4,3.1,Encoder and Decoder Stacks,3
5,3.2,Attention,3
6,3.2.1,Scaled Dot-Product Attention,4
7,3.2.2,Multi-Head Attention,4
8,3.2.3,Applications of Attention in our Model,5
9,3.3,Position-wise Feed-Forward Networks,5


---
## Task 3 — Paragraph Detection

The paragraph detector uses **PyMuPDF layout blocks** as candidates, then applies several filters:

**How paragraph boundaries are determined:**
1. Each PDF text block becomes a candidate paragraph
2. Recognized headings (from Task 2) are excluded and used to assign section labels
3. Captions (`Figure 1:`, `Table 2:`) are excluded using regex patterns
4. Page numbers, equations, and short non-prose fragments are filtered out
5. Front-matter blocks on page 1 (title, authors, affiliations) before the first heading are excluded
6. Footer/footnote blocks at the bottom of pages with smaller font size are excluded
7. Adjacent blocks on the same page, column, and section are **merged** when the previous block lacks sentence-ending punctuation and the next block starts with a lowercase letter
8. References and Acknowledgments sections are excluded from the body paragraphs

All excluded blocks and their reasons are retained in the output for auditing.

In [7]:
from scipaper.paragraphs import extract_document_paragraphs
import json
import pandas as pd

paragraph_dir = PROJECT_ROOT / "outputs" / "paragraphs"
paragraph_dir.mkdir(parents=True, exist_ok=True)

paragraph_summaries = []
para_rows = []
for pdf_path in PDF_FILES:
    result = extract_document_paragraphs(pdf_path)
    output_path = paragraph_dir / f"{pdf_path.stem}_paragraphs.json"
    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    stats = result["statistics"]
    paragraph_summaries.append({"paper": result["paper"], **stats, "output": str(output_path)})
    para_rows.append({
        "Paper": result["paper"],
        "Pages": stats["pages"],
        "Sections detected": stats["detected_sections"],
        "Paragraphs": stats["paragraphs"],
        "Excluded candidates": stats["excluded_candidates"],
    })

total_paragraphs = sum(s["paragraphs"] for s in paragraph_summaries)
para_rows.append({
    "Paper": "TOTAL",
    "Pages": sum(r["Pages"] for r in para_rows),
    "Sections detected": sum(r["Sections detected"] for r in para_rows),
    "Paragraphs": total_paragraphs,
    "Excluded candidates": sum(r["Excluded candidates"] for r in para_rows),
})
display(pd.DataFrame(para_rows))


,Paper,Pages,Sections detected,Paragraphs,Excluded candidates
0,1406.1078v3,15,23,152,275
1,1409.0473v7,15,33,119,327
2,1412.3555v1,9,14,86,160
3,1601.06733v7,11,14,82,162
4,1607.06450v1,14,22,131,468
5,1610.02357v3,8,19,68,53
6,1703.03906v2,9,20,78,111
7,AttentionYouNeed,15,25,85,454
8,N16-1024,11,22,127,105
9,TOTAL,107,192,928,2115


In [8]:
# Show example paragraphs from the main paper
result = extract_document_paragraphs(PAPER_DIR / 'AttentionYouNeed.pdf')
print("First 5 paragraph records from Attention Is All You Need:\n")
for para in result['paragraphs'][:5]:
    section = f"{para.get('section_id', '')} {para.get('section', 'Unassigned')}".strip()
    print(f"  P{para['paragraph']} | page {para['page']} | {section}")
    print(f"    {para['text'][:120]}...")
    print()

First 5 paragraph records from Attention Is All You Need:

  P1 | page 1 | None Abstract
    The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include a...

  P2 | page 1 | None Abstract
    ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort t...

  P3 | page 1 | None Abstract
    31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA....

  P4 | page 2 | 1 Introduction
    Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been ...

  P5 | page 2 | 1 Introduction
    Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the...



---
## Task 4 — Sentence Segmentation

We use **spaCy's trained English parser** (`en_core_web_sm`) for sentence boundary detection, plus a narrow abbreviation repair rule.

### Why `text.split(".")` is inadequate for scientific papers

`text.split(".")` treats every period as a sentence boundary. This fails in at least five common cases in scientific text:

1. **Abbreviations:** `"et al."` would be split into `"et al"` + `""`, breaking citation patterns like `"Vaswani et al. proposed..."` into two fragments.

2. **Decimal numbers:** `"achieves 28.4 BLEU"` would be split at the period, producing `"achieves 28"` and `"4 BLEU"` — two meaningless fragments from one sentence.

3. **Citations and references:** `"See Sec. 3.2 for details"` contains three periods, none of which is a sentence boundary. `split(".")` would produce four fragments.

4. **Parenthetical expressions:** `"(Devlin et al., 2014)"` would split inside the parentheses, breaking the citation structure.

5. **Equations and formulas:** `"We compute Eq. (1) where d_k = 64."` would split after `"Eq"` rather than at the actual sentence end.

Our pipeline uses spaCy's parser, which considers syntactic structure, and adds a conservative repair for known abbreviations followed by parentheses (e.g., joining `"et al."` + `"(2017)"` when the parser incorrectly splits them).

In [9]:
import spacy
from scipaper.sentences import process_file

nlp = spacy.load("en_core_web_sm")

# Demonstrate why text.split(".") fails
demo_text = "Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task. See Sec. 3.2 for details."
print("Demo sentence:", demo_text)
print(f"\ntext.split('.') produces {len(demo_text.split('.'))} fragments:")
for i, frag in enumerate(demo_text.split('.')):
    print(f"  Fragment {i+1}: '{frag.strip()}'")

print(f"\nspaCy parser produces {len(list(nlp(demo_text).sents))} sentences:")
for i, sent in enumerate(nlp(demo_text).sents):
    print(f"  Sentence {i+1}: '{sent.text.strip()}'")

Demo sentence: Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task. See Sec. 3.2 for details.

text.split('.') produces 7 fragments:
  Fragment 1: 'Vaswani et al'
  Fragment 2: '(2017) achieve 28'
  Fragment 3: '4 BLEU on the WMT 2014 task'
  Fragment 4: 'See Sec'
  Fragment 5: '3'
  Fragment 6: '2 for details'
  Fragment 7: ''

spaCy parser produces 2 sentences:
  Sentence 1: 'Vaswani et al. (2017) achieve 28.4 BLEU on the WMT 2014 task.'
  Sentence 2: 'See Sec. 3.2 for details.'


In [10]:
# Run Task 4 on all paragraph JSON files
import pandas as pd

paragraph_dir = PROJECT_ROOT / "outputs" / "paragraphs"
sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
paragraph_jsons = sorted(paragraph_dir.glob("*_paragraphs.json"))

sentence_results = [process_file(path, nlp, sentence_dir) for path in paragraph_jsons]

sent_rows = [
    {"Paper": r["paper"], "Paragraphs": r["paragraphs"], "Sentences": r["sentences"]}
    for r in sentence_results
]
sent_rows.append({
    "Paper": "TOTAL",
    "Paragraphs": sum(r["paragraphs"] for r in sentence_results),
    "Sentences": sum(r["sentences"] for r in sentence_results),
})
display(pd.DataFrame(sent_rows))


,Paper,Paragraphs,Sentences
0,1406.1078v3,152,328
1,1409.0473v7,119,276
2,1412.3555v1,86,185
3,1601.06733v7,82,250
4,1607.06450v1,131,330
5,1610.02357v3,68,186
6,1703.03906v2,78,196
7,AttentionYouNeed,85,233
8,N16-1024,127,301
9,TOTAL,928,2285


---
## Task 5 — Tokenization and Linguistic Annotations

Each corrected Task 4 sentence is parsed independently by spaCy. For every token we record:
- **text** — the surface form
- **token_index** — zero-based position within the sentence
- **lemma** — the dictionary base form
- **pos** — coarse part-of-speech tag
- **dependency** — syntactic dependency relation
- **head** — the syntactic head token

We also add a small tokenizer extension: two infix rules that split parentheses attached to letters (e.g., `p(xt)` → `p`, `(`, `xt`, `)`). This improves token boundaries around inline mathematical notation without affecting other English tokenization.

In [11]:
import spacy
import pandas as pd
from scipaper.tokens import configure_scientific_tokenizer, process_file as token_process_file

task5_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(task5_nlp)

sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
token_dir = PROJECT_ROOT / "outputs" / "tokens"
sentence_jsons = sorted(sentence_dir.glob("*_sentences.json"))

token_results = [token_process_file(path, task5_nlp, token_dir) for path in sentence_jsons]

tok_rows = [
    {"Paper": r["paper"], "Sentences": r["sentences"], "Tokens (incl. punct)": r["tokens"]}
    for r in token_results
]
tok_rows.append({
    "Paper": "TOTAL",
    "Sentences": sum(r["sentences"] for r in token_results),
    "Tokens (incl. punct)": sum(r["tokens"] for r in token_results),
})
display(pd.DataFrame(tok_rows))


,Paper,Sentences,Tokens (incl. punct)
0,1406.1078v3,328,6670
1,1409.0473v7,276,6659
2,1412.3555v1,185,3512
3,1601.06733v7,250,5746
4,1607.06450v1,330,5996
5,1610.02357v3,186,3962
6,1703.03906v2,196,4476
7,AttentionYouNeed,233,4895
8,N16-1024,301,6152
9,TOTAL,2285,48068


In [12]:
# Show token example from the first sentence of Attention Is All You Need
import json
import pandas as pd

token_path = token_dir / "AttentionYouNeed_tokens.json"
token_doc = json.loads(token_path.read_text(encoding="utf-8"))
first_sentence = token_doc["paragraphs"][0]["sentences"][0]

print("Sentence:", first_sentence["text"])
print()
tok_rows = [
    {
        "Token": tok["text"],
        "Lemma": tok["lemma"],
        "POS": tok["pos"],
        "Dependency": tok["dependency"],
        "Head": tok["head"],
    }
    for tok in first_sentence["tokens"][:12]
]
display(pd.DataFrame(tok_rows))
if len(first_sentence["tokens"]) > 12:
    print(f"... ({len(first_sentence['tokens'])} tokens total)")


Sentence: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.



,Token,Lemma,POS,Dependency,Head
0,The,the,DET,det,models
1,dominant,dominant,ADJ,amod,models
2,sequence,sequence,NOUN,compound,transduction
3,transduction,transduction,NOUN,compound,models
4,models,model,NOUN,nsubjpass,based
5,are,be,AUX,auxpass,based
6,based,base,VERB,ROOT,based
7,on,on,ADP,prep,based
8,complex,complex,ADJ,amod,recurrent
9,recurrent,recurrent,NOUN,pobj,on


... (22 tokens total)


---
## Task 6 — Part-of-Speech Analysis

POS statistics use non-punctuation tokens as the denominator (punctuation is excluded from the nine required POS categories but its count is reported separately). We compare three papers to observe differences in their linguistic profiles.

In [13]:
import pandas as pd
from scipaper.pos_stats import process_files

token_dir = PROJECT_ROOT / "outputs" / "tokens"
statistics_dir = PROJECT_ROOT / "outputs" / "statistics"
token_jsons = sorted(token_dir.glob("*_tokens.json"))
pos_rows = process_files(token_jsons, statistics_dir)

REQUIRED_POS = ("NOUN", "VERB", "ADJ", "ADV", "PROPN", "PRON", "DET", "ADP", "NUM")

# All-paper required POS percentages
all_paper_rows = []
for row in pos_rows:
    entry = {"Paper": row["paper"], "Non-punct tokens": row["pos_percentage_denominator"]}
    for pos in REQUIRED_POS:
        entry[pos] = f"{row['pos_percentages'][pos]:.1f}%"
    all_paper_rows.append(entry)

print(f"Required POS percentages — all {len(all_paper_rows)} papers")
display(pd.DataFrame(all_paper_rows))

# Three-paper detailed comparison: counts + percentages
selected_papers = {"AttentionYouNeed", "N16-1024", "1607.06450v1"}
comparison_rows = []
for row in pos_rows:
    if row["paper"] in selected_papers:
        for pos in REQUIRED_POS:
            comparison_rows.append({
                "Paper": row["paper"],
                "POS": pos,
                "Count": row["pos_counts"][pos],
                "Percentage": f"{row['pos_percentages'][pos]:.2f}%",
            })

print("\nDetailed POS count and percentage comparison — three papers")
display(pd.DataFrame(comparison_rows))


Required POS percentages — all 9 papers


,Paper,Non-punct tokens,NOUN,VERB,ADJ,ADV,PROPN,PRON,DET,ADP,NUM
0,1406.1078v3,5639,24.0%,10.9%,7.3%,3.4%,12.0%,3.4%,12.9%,10.8%,3.0%
1,1409.0473v7,5598,24.8%,11.4%,7.4%,3.7%,9.9%,3.6%,14.2%,11.2%,2.8%
2,1412.3555v1,2971,23.9%,9.9%,9.8%,4.5%,9.0%,3.7%,12.8%,11.3%,3.2%
3,1601.06733v7,4871,26.5%,10.7%,9.5%,3.0%,10.4%,4.1%,9.7%,11.1%,4.5%
4,1607.06450v1,5282,27.9%,11.1%,7.7%,2.9%,8.3%,3.2%,11.5%,11.2%,4.1%
5,1610.02357v3,3508,25.4%,9.4%,11.0%,3.9%,7.3%,4.1%,9.4%,13.1%,4.6%
6,1703.03906v2,3869,25.8%,11.5%,9.5%,4.3%,9.6%,5.1%,8.2%,11.0%,3.6%
7,AttentionYouNeed,4213,29.1%,10.5%,9.1%,3.4%,5.1%,4.1%,10.4%,12.6%,5.5%
8,N16-1024,5032,27.4%,10.2%,8.8%,3.3%,11.8%,3.9%,9.9%,10.4%,3.5%



Detailed POS count and percentage comparison — three papers


,Paper,POS,Count,Percentage
0,1607.06450v1,NOUN,1475,27.93%
1,1607.06450v1,VERB,588,11.13%
2,1607.06450v1,ADJ,409,7.74%
3,1607.06450v1,ADV,155,2.93%
4,1607.06450v1,PROPN,440,8.33%
5,1607.06450v1,PRON,171,3.24%
6,1607.06450v1,DET,606,11.47%
7,1607.06450v1,ADP,589,11.15%
8,1607.06450v1,NUM,214,4.05%
9,AttentionYouNeed,NOUN,1225,29.08%


### Three-Paper POS Comparison

The three papers show broadly similar POS distributions — as expected for scientific English — but with notable differences:

- **NOUN** is the most frequent category in all three papers (~27–29%), reflecting the noun-heavy nature of scientific writing
- **PROPN (proper nouns)** varies significantly: `N16-1024` has the highest share because it discusses many named systems, models, and benchmarks
- **NUM** is highest in `AttentionYouNeed` because of its extensive results tables with BLEU scores and training parameters
- **VERB** is relatively low (~10–11%) compared to general English, as scientific papers favor nominal constructions

These differences reflect the content and style of each paper rather than fundamental linguistic properties.

---
## Task 7 — Dependency Parsing and Visualization

We selected 10 sentences from 10 distinct sections across all nine papers. For each sentence, we display token, lemma, POS, dependency label, and syntactic head. Three sentences are visualized with displaCy and their structures are briefly explained.

In [14]:
import json
import pandas as pd
import spacy
from IPython.display import HTML, display
from scipaper.dependencies import build_dependency_examples
from scipaper.tokens import configure_scientific_tokenizer

dependency_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(dependency_nlp)

dependency_manifest = PROJECT_ROOT / "data" / "annotations" / "task7_dependency_examples.json"
dependency_output_dir = PROJECT_ROOT / "outputs" / "dependency_examples"

dependency_summary = build_dependency_examples(
    dependency_manifest,
    PROJECT_ROOT / "outputs" / "tokens",
    dependency_output_dir,
    dependency_nlp,
)

summary_rows = [
    {"Metric": "Dependency examples", "Value": dependency_summary["samples"]},
    {"Metric": "Distinct sections", "Value": dependency_summary["distinct_sections"]},
    {"Metric": "Token rows in CSV", "Value": dependency_summary["token_rows"]},
    {"Metric": "displaCy visualizations", "Value": len(dependency_summary["visualizations"])},
]
display(pd.DataFrame(summary_rows))


,Metric,Value
0,Dependency examples,10
1,Distinct sections,10
2,Token rows in CSV,238
3,displaCy visualizations,3


In [15]:
# Display all dependency-token rows, grouped by sentence to avoid pandas truncation
import json
from IPython.display import HTML, display

examples_path = dependency_output_dir / "dependency_examples.json"
dependency_examples = json.loads(examples_path.read_text(encoding="utf-8"))["examples"]
example_by_id = {example["case_id"]: example for example in dependency_examples}
dependency_csv = dependency_output_dir / "dependency_tokens.csv"
dep_table = pd.read_csv(dependency_csv)
token_columns = ["token", "lemma", "pos", "dependency", "head"]

for case_id, case_tokens in dep_table.groupby("case_id", sort=False):
    example = example_by_id[case_id]
    print(f"{case_id} — {example['paper']} — {example['section']}")
    print(f"Sentence: {example['text']}")
    display(HTML(case_tokens[token_columns].to_html(index=False, border=0)))


passive_training_objective — 1406.1078v3 — Abstract
Sentence: The encoder and decoder of the proposed model are jointly trained to maximize the conditional probability of a target sequence given a source sequence.


token,lemma,pos,dependency,head
The,the,DET,det,encoder
encoder,encoder,NOUN,nsubjpass,trained
and,and,CCONJ,cc,encoder
decoder,decoder,NOUN,conj,encoder
of,of,ADP,prep,encoder
the,the,DET,det,model
proposed,propose,VERB,amod,model
model,model,NOUN,pobj,of
are,be,AUX,auxpass,trained
jointly,jointly,ADV,advmod,trained


translation_experiment — 1406.1078v3 — Experiments
Sentence: We evaluate our approach on the English/French translation task of the WMT’14 workshop.


token,lemma,pos,dependency,head
We,we,PRON,nsubj,evaluate
evaluate,evaluate,VERB,ROOT,evaluate
our,our,PRON,poss,approach
approach,approach,NOUN,dobj,evaluate
on,on,ADP,prep,approach
the,the,DET,det,task
English,English,PROPN,nmod,French
/,/,SYM,punct,French
French,french,ADJ,amod,task
translation,translation,NOUN,compound,task


neural_machine_translation_background — 1409.0473v7 — BACKGROUND: NEURAL MACHINE TRANSLATION
Sentence: From a probabilistic perspective, translation is equivalent to ﬁnding a target sentence y that maximizes the conditional probability of y given a source sentence x, i.e., arg maxy p(y | x).


...

token,lemma,pos,dependency,head
We,we,PRON,nsubj,report
report,report,VERB,ROOT,report
BLEU,BLEU,PROPN,compound,scores
scores,score,NOUN,dobj,report
",",",",PUNCT,punct,scores
perplexities,perplexity,NOUN,conj,scores
",",",",PUNCT,punct,perplexities
model,model,NOUN,compound,sizes
sizes,size,NOUN,conj,perplexities
",",",",PUNCT,punct,sizes


encoder_representations — AttentionYouNeed — Model Architecture
Sentence: Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence of continuous representations z = (z1, ..., zn).


token,lemma,pos,dependency,head
Here,here,ADV,advmod,maps
",",",",PUNCT,punct,maps
the,the,DET,det,encoder
encoder,encoder,NOUN,nsubj,maps
maps,map,VERB,ROOT,maps
an,an,DET,det,sequence
input,input,NOUN,compound,sequence
sequence,sequence,NOUN,dobj,maps
of,of,ADP,prep,sequence
symbol,symbol,NOUN,compound,representations


transition_sequence — N16-1024 — Transition Sequences from Trees
Sentence: Any parse tree can be converted to a sequence of transitions via a depth-ﬁrst, left-to-right traversal of a parse tree.


token,lemma,pos,dependency,head
Any,any,DET,det,tree
parse,parse,NOUN,compound,tree
tree,tree,NOUN,nsubjpass,converted
can,can,AUX,aux,converted
be,be,AUX,auxpass,converted
converted,convert,VERB,ROOT,converted
to,to,ADP,prep,converted
a,a,DET,det,sequence
sequence,sequence,NOUN,pobj,to
of,of,ADP,prep,sequence


### Three Dependency Structure Explanations

**1. Passive training objective** (`1406.1078v3`, Abstract):
The sentence uses a passive construction where `"trained"` is the ROOT verb. The encoder and decoder are coordinated passive subjects (`nsubjpass`), joined by `"and"` (cc/conj). The infinitive `"maximize"` functions as an adverbial complement of purpose (xcomp), and `"probability"` is its direct object (dobj). This tree shows how spaCy correctly identifies the passive voice and purpose clause.

**2. Relative clause** (`1601.06733v7`, The Machine Reader):
`"present"` is the ROOT, with `"we"` as its subject (nsubj) and `"reader"` as its direct object (dobj). The relative clause `"which is designed to..."` modifies `"reader"` — `"which"` is the subject (nsubjpass) of the passive `"designed"`, and `"process"` is an open clausal complement (xcomp). The diagram illustrates how spaCy handles relative clause attachment.

**3. Coordinated measures** (`1703.03906v2`, Introduction):
`"report"` is the ROOT with `"We"` as subject. The four reported measures form a coordinated list joined by commas and `"and"`. spaCy may attach list items imperfectly — this is a known limitation with long coordinated structures. The participial phrase `"including variance numbers"` attaches as an adverbial modifier.

In [16]:
# Display displaCy visualizations
from spacy import displacy
from IPython.display import HTML

manifest = json.loads(dependency_manifest.read_text(encoding="utf-8"))
visualize_cases = [c for c in manifest["cases"] if c.get("visualize")]

for case in visualize_cases:
    # Read the sentence text from the dependency examples output
    dep_examples = json.loads((dependency_output_dir / "dependency_examples.json").read_text(encoding="utf-8"))
    example = next(e for e in dep_examples["examples"] if e["case_id"] == case["case_id"])
    
    print(f"\n{'='*80}")
    print(f"Case: {case['case_id']} | Paper: {case['paper']} | Section: {case['section']}")
    print(f"Sentence: {example['text']}")
    print(f"{'='*80}")
    
    doc = dependency_nlp(example["text"])
    html = displacy.render(doc, style="dep", jupyter=False, options={"compact": True, "distance": 100})
    display(HTML(html))


Case: passive_training_objective | Paper: 1406.1078v3 | Section: Abstract
Sentence: The encoder and decoder of the proposed model are jointly trained to maximize the conditional probability of a target sequence given a source sequence.



Case: machine_reader_relative_clause | Paper: 1601.06733v7 | Section: The Machine Reader
Sentence: In this section we present our machine reader which is designed to process structured input while retaining the incrementality of a recurrent neural network.



Case: reporting_experimental_results | Paper: 1703.03906v2 | Section: Introduction
Sentence: We report BLEU scores, perplexities, model sizes, and convergence time for all experiments, including variance numbers calculated across several runs of each experiment.


---
## Task 8 — Document Statistics

The table below summarizes key metrics for each paper. Definitions:
- **Tokens** includes punctuation (total spaCy tokens)
- **Unique Tokens** are case-folded surface forms excluding punctuation
- **Avg Sentence Length** and **Avg Paragraph Length** count non-punctuation tokens

In [17]:
import pandas as pd
from scipaper.document_stats import process_files

token_files = sorted((PROJECT_ROOT / "outputs" / "tokens").glob("*_tokens.json"))
statistics_dir = PROJECT_ROOT / "outputs" / "statistics"
document_stats_rows = process_files(token_files, statistics_dir)

df = pd.DataFrame(document_stats_rows)
df.columns = [c.replace("_", " ").title() for c in df.columns]

# Keep the required cross-paper comparison narrow enough to read in VS Code.
required_columns = [
    "Paper", "Pages", "Sections Detected", "Paragraphs", "Sentences",
    "Tokens Including Punctuation",
]
print(f"Required document statistics — all {len(df)} papers")
display(df[required_columns])

# The assignment also asks for lexical diversity and average lengths.
extra_columns = {
    "Paper": "Paper",
    "Unique Tokens Casefolded Excluding Punctuation": "Unique lexical forms",
    "Average Sentence Length Lexical Tokens": "Avg. lexical tokens / sentence",
    "Average Paragraph Length Lexical Tokens": "Avg. lexical tokens / paragraph",
}
print("Additional vocabulary and length statistics — all papers")
display(df[list(extra_columns)].rename(columns=extra_columns))

print(f"\n{'='*60}")
print(f"TOTALS across {len(df)} papers:")
print(f"  Pages:      {df['Pages'].sum()}")
print(f"  Paragraphs: {df['Paragraphs'].sum()}")
print(f"  Sentences:  {df['Sentences'].sum()}")
print(f"  Tokens:     {df['Tokens Including Punctuation'].sum():,}")


Required document statistics — all 9 papers


,Paper,Pages,Sections Detected,Paragraphs,Sentences,Tokens Including Punctuation
0,1406.1078v3,15,23,152,328,6670
1,1409.0473v7,15,33,119,276,6659
2,1412.3555v1,9,14,86,185,3512
3,1601.06733v7,11,14,82,250,5746
4,1607.06450v1,14,22,131,330,5996
5,1610.02357v3,8,19,68,186,3962
6,1703.03906v2,9,20,78,196,4476
7,AttentionYouNeed,15,25,85,233,4895
8,N16-1024,11,22,127,301,6152


Additional vocabulary and length statistics — all papers


,Paper,Unique lexical forms,Avg. lexical tokens / sentence,Avg. lexical tokens / paragraph
0,1406.1078v3,1132,17.19,37.10
1,1409.0473v7,1135,20.28,47.04
2,1412.3555v1,708,16.06,34.55
3,1601.06733v7,1237,19.48,59.40
4,1607.06450v1,1142,16.01,40.32
5,1610.02357v3,839,18.86,51.59
6,1703.03906v2,1030,19.74,49.60
7,AttentionYouNeed,1075,18.08,49.56
8,N16-1024,1120,16.72,39.62



TOTALS across 9 papers:
  Pages:      107
  Paragraphs: 928
  Sentences:  2285
  Tokens:     48,068


---
## Task 9 — Final Nested JSON Representation

The pipeline produces one structured JSON file per paper with the hierarchy:

```
Paper
  → Section (section_id, title)
    → Paragraph (paragraph_id, page, text)
      → Sentence (sentence_id, text)
        → Token (text, token_index, lemma, pos, dependency, head)
```

This output will be used as input for Homework 2.

In [18]:
import pandas as pd
from scipaper.final_json import process_files as build_final_json

token_files = sorted((PROJECT_ROOT / "outputs" / "tokens").glob("*_tokens.json"))
final_summaries = build_final_json(token_files, PROJECT_ROOT / "outputs" / "json")

final_rows = [
    {
        "Paper": s["paper"],
        "Sections (with paragraphs)": s["sections_with_paragraphs"],
        "Paragraphs": s["paragraphs"],
        "Sentences": s["sentences"],
        "Tokens": s["tokens"],
    }
    for s in final_summaries
]
final_rows.append({
    "Paper": "TOTAL",
    "Sections (with paragraphs)": sum(r["Sections (with paragraphs)"] for r in final_rows),
    "Paragraphs": sum(r["Paragraphs"] for r in final_rows),
    "Sentences": sum(r["Sentences"] for r in final_rows),
    "Tokens": sum(r["Tokens"] for r in final_rows),
})
display(pd.DataFrame(final_rows))


,Paper,Sections (with paragraphs),Paragraphs,Sentences,Tokens
0,1406.1078v3,20,152,328,6670
1,1409.0473v7,23,119,276,6659
2,1412.3555v1,10,86,185,3512
3,1601.06733v7,12,82,250,5746
4,1607.06450v1,19,131,330,5996
5,1610.02357v3,15,68,186,3962
6,1703.03906v2,16,78,196,4476
7,AttentionYouNeed,22,85,233,4895
8,N16-1024,20,127,301,6152
9,TOTAL,157,928,2285,48068


In [19]:
# Preview the structure of the main paper's JSON
import json
import pandas as pd

preview_path = PROJECT_ROOT / "outputs" / "json" / "AttentionYouNeed.json"
attention_json = json.loads(preview_path.read_text(encoding="utf-8"))

print(f"Paper: {attention_json['paper']}  |  Schema version: {attention_json['schema_version']}")

section_rows = []
for sec in attention_json["sections"]:
    para_count = len(sec["paragraphs"])
    sent_count = sum(len(p["sentences"]) for p in sec["paragraphs"])
    section_rows.append({
        "Section": sec["title"],
        "Paragraphs": para_count,
        "Sentences": sent_count,
    })
print(f"\nSection breakdown — AttentionYouNeed ({len(section_rows)} sections):")
display(pd.DataFrame(section_rows))

print("\n--- First sentence of first section (preview) ---")
first_sent = attention_json["sections"][0]["paragraphs"][0]["sentences"][0]
print(f"Text: {first_sent['text'][:120]}...")
print(f"Tokens: {len(first_sent['tokens'])} tokens")
print(f"First 5 tokens: {[t['text'] for t in first_sent['tokens'][:5]]}")


Paper: AttentionYouNeed  |  Schema version: 1.0

Section breakdown — AttentionYouNeed (22 sections):


,Section,Paragraphs,Sentences
0,Abstract,3,16
1,1 Introduction,4,13
2,2 Background,4,12
3,3 Model Architecture,2,6
4,3.1 Encoder and Decoder Stacks,2,12
5,3.2 Attention,4,6
6,3.2.1 Scaled Dot-Product Attention,4,14
7,3.2.2 Multi-Head Attention,8,15
8,3.2.3 Applications of Attention in our Model,1,1
9,3.3 Position-wise Feed-Forward Networks,5,16



--- First sentence of first section (preview) ---
Text: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include a...
Tokens: 22 tokens
First 5 tokens: ['The', 'dominant', 'sequence', 'transduction', 'models']


---
## Summary

This notebook processed **9 scientific PDFs** through a complete NLP pipeline:

| Stage | Output |
|---|---|
| Task 1 — PDF Extraction | Page-marked text files |
| Task 2 — Section Detection | Heading candidates with page numbers |
| Task 3 — Paragraph Detection | Filtered, section-labeled paragraph records |
| Task 4 — Sentence Segmentation | spaCy sentences with abbreviation repair |
| Task 5 — Tokenization | Tokens with lemma, POS, dependency, head |
| Task 6 — POS Analysis | Per-paper POS counts and percentages |
| Task 7 — Dependency Parsing | 10 examples, 3 displaCy visualizations |
| Task 8 — Document Statistics | Cross-paper comparison table |
| Task 9 — JSON Output | One nested JSON file per paper |

All outputs are in `outputs/`. The experiment log (`notes/experiment_log.md`) documents the methods tried, what worked, and what didn't at each stage.